# Document Question Answering System (RAG)
### Celebal Technologies — Week 7 Project

This notebook implements a **Retrieval-Augmented Generation (RAG)** system that answers questions
based on a custom document (a resume, in this case), following the architecture below:

1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database
5. Query Processing
6. Context Retrieval
7. Answer Generation

Instead of relying only on a language model's internal knowledge, the system retrieves relevant
information from the document and generates answers grounded in that information.


## Setup
Install the required libraries. All components used here (embedding model, vector store, language model) are free and run fully offline — no API key required.

In [2]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 26.7 MB/s eta 0:00:00


In [3]:
import os
import re
import textwrap

import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
def load_pdf_text(pdf_path: str) -> str:
    """
    Loads a PDF from `pdf_path` and extracts all text.
    """
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

## 1. Document Ingestion
Documents such as PDFs or text files are loaded and converted into raw text.

We use a resume PDF as our custom document — RAG is meant for exactly this kind of
private/custom data that a general-purpose language model wouldn't know about.


### Upload your PDF

To use your own PDF, upload it to your Colab environment. You can do this by clicking the folder icon on the left sidebar, then clicking the 'Upload' icon (a page with an arrow pointing up). Once uploaded, you'll see the file in the file browser. Copy its path and update the `PDF_PATH` variable in the cell below.

In [6]:
import requests

GITHUB_RAW_URL = "https://raw.githubusercontent.com/Skyline40-k/CODSOFT/main/Abhay_Ranjan_Enhanced_Resume.pdf"

response = requests.get(GITHUB_RAW_URL)
with open("resume.pdf", "wb") as f:
    f.write(response.content)
print("Resume downloaded from GitHub ✓")

PDF_PATH = "resume.pdf"
document_text = load_pdf_text(PDF_PATH)
print(f"Document loaded. Total characters: {len(document_text)}")

Resume downloaded from GitHub ✓
Document loaded. Total characters: 1676


## 2. Text Chunking
The text is split into smaller chunks to improve retrieval accuracy. Smaller chunks make it
easier for the embedding model to capture focused, specific meaning rather than diluting it
across an entire page.


In [7]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list:
    """
    Splits text into overlapping chunks of roughly `chunk_size` characters.
    Overlap helps avoid losing context that falls on a chunk boundary.
    """
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(document_text, chunk_size=300, overlap=50)
print(f"Document split into {len(chunks)} chunks.\n")
for i, c in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ---")
    print(textwrap.fill(c, 100))
    print()

Document split into 7 chunks.

--- Chunk 0 ---
Abhay Ranjan Java Backend Developer | MERN Stack Developer | Problem Solver +91 8294986686 |
abhayranjan404@gmail.com | LinkedIn | GitHub | LeetCode Professional Summary Computer Science
undergraduate with a strong interest in backend development, data structures and algorithms, and
scalable web app

--- Chunk 1 ---
ta structures and algorithms, and scalable web applications. Experienced with Java, JavaScript,
Node.js, Express.js, React.js, Spring Framework, MongoDB, and MySQL. Quick learner with strong
analytical thinking, problem-solving ability, and experience working in collaborative environments
through ha

--- Chunk 2 ---
e working in collaborative environments through hackathons, projects, and student leadership.
Education Maharishi Markandeshwar (Deemed to be University) B.Tech in Computer Science & Engineering
(2023–2027) CGPA: 8.6/10 Experience Student Coordinator – Alpha Intern, MMDU Coordinated campus
events, m



## 3. Embedding Creation
Each chunk is converted into a vector representation capturing its semantic meaning.
We use `all-MiniLM-L6-v2`, a small, fast sentence-transformer model that runs comfortably
on CPU — no GPU or API key needed.


In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
print(f"Created embeddings of shape: {chunk_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created embeddings of shape: (7, 384)


## 4. Vector Database
Embeddings are stored in a vector database (here, FAISS) for efficient similarity search.
FAISS lets us quickly find which chunks are closest in meaning to a given query.


In [9]:
embedding_dim = chunk_embeddings.shape[1]

# L2-normalize so that inner product search behaves like cosine similarity
faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print(f"Vector database built with {index.ntotal} chunk embeddings.")

Vector database built with 7 chunk embeddings.


## 5. Query Processing
The user's question is converted into an embedding using the same embedding model,
so it lives in the same vector space as the document chunks.


In [10]:
def embed_query(question: str) -> np.ndarray:
    q_embedding = embedding_model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_embedding)
    return q_embedding

## 6. Context Retrieval
The system retrieves the most relevant chunks from the vector database using similarity search.


In [11]:
def retrieve_context(question: str, top_k: int = 3) -> list:
    """Returns the top_k most relevant chunks for a given question."""
    q_embedding = embed_query(question)
    scores, indices = index.search(q_embedding, top_k)
    retrieved = [(chunks[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]
    return retrieved

# Quick test
test_question = "What is Abhay's CGPA?"
retrieved_chunks = retrieve_context(test_question, top_k=3)
for chunk, score in retrieved_chunks:
    print(f"(score={score:.3f}) {textwrap.fill(chunk, 100)}\n")

(score=0.397) e working in collaborative environments through hackathons, projects, and student leadership.
Education Maharishi Markandeshwar (Deemed to be University) B.Tech in Computer Science & Engineering
(2023–2027) CGPA: 8.6/10 Experience Student Coordinator – Alpha Intern, MMDU Coordinated campus
events, m

(score=0.224) ication (2025)  Frontend Web Developer Certification – Infosys Springboard (2025)  Finalist –
Internal Smart India Hackathon 2025  Participant – Hackureka Hackathon (GDSC)

(score=0.158) Abhay Ranjan Java Backend Developer | MERN Stack Developer | Problem Solver +91 8294986686 |
abhayranjan404@gmail.com | LinkedIn | GitHub | LeetCode Professional Summary Computer Science
undergraduate with a strong interest in backend development, data structures and algorithms, and
scalable web app



## 7. Answer Generation
A language model generates the final answer using the retrieved context, ensuring responses
are grounded in actual data rather than the model's internal (and possibly outdated or
incorrect) knowledge.

We use `google/flan-t5-base`, a small open-source instruction-tuned model that runs on CPU
without needing an API key.


In [12]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.eval()

def generate_answer(question: str, top_k: int = 3, max_new_tokens: int = 128) -> str:
    retrieved = retrieve_context(question, top_k=top_k)
    context = "\n".join(chunk for chunk, _ in retrieved)

    prompt = (
        "Answer the question using only the context below. "
        "If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, retrieved

print("Generation model loaded successfully.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generation model loaded successfully.


## Full Pipeline in Action
This ties together the entire workflow described in the project document:

1. Load and preprocess document
2. Split text into chunks
3. Convert chunks into embeddings
4. Store embeddings in a vector database
5. Accept user query
6. Retrieve relevant chunks
7. Generate answer using retrieved context


In [13]:
def ask(question: str, top_k: int = 3, show_context: bool = True):
    answer, retrieved = generate_answer(question, top_k=top_k)
    print(f"Question: {question}\n")
    print(f"Answer: {answer}\n")
    if show_context:
        print("Retrieved context used:")
        for chunk, score in retrieved:
            print(f"  (score={score:.3f}) {textwrap.fill(chunk, 90)}\n")
    print("=" * 100)

# Example Flow (matches the example in the project document)
ask("What is the main idea of the document?")

Question: What is the main idea of the document?

Answer: Integrated OCR to extract ingredients from food labels. Designed backend APIs and database for ingredient classification and health analysis.

Retrieved context used:
  (score=0.178) t, Dart Frameworks: React.js, Node.js, Express.js, Spring Framework Databases: MongoDB,
MySQL Core CS: DSA, OOP, DBMS, Operating Systems, Computer Networks Tools: Git, GitHub, VS
Code Certifications & Achievements  Principles of Generative AI Certification (2025) 
Frontend Web Developer Certificat

  (score=0.163) ta structures and algorithms, and scalable web applications. Experienced with Java,
JavaScript, Node.js, Express.js, React.js, Spring Framework, MongoDB, and MySQL. Quick
learner with strong analytical thinking, problem-solving ability, and experience working
in collaborative environments through ha

  (score=0.117) eb application using React, Node.js, Express.js and MongoDB.  Integrated OCR to extract
ingredients from food labels.  De

In [14]:
# A few more example questions to demonstrate retrieval + generation quality
ask("Where is Abhay currently doing his internship?")
ask("What programming languages does Abhay know?")
ask("What is Abhay's CGPA?")
ask("List two projects Abhay has built.")

Question: Where is Abhay currently doing his internship?

Answer: Maharishi Markandeshwar

Retrieved context used:
  (score=0.435) e working in collaborative environments through hackathons, projects, and student
leadership. Education Maharishi Markandeshwar (Deemed to be University) B.Tech in Computer
Science & Engineering (2023–2027) CGPA: 8.6/10 Experience Student Coordinator – Alpha
Intern, MMDU Coordinated campus events, m

  (score=0.335) Abhay Ranjan Java Backend Developer | MERN Stack Developer | Problem Solver +91 8294986686
| abhayranjan404@gmail.com | LinkedIn | GitHub | LeetCode Professional Summary Computer
Science undergraduate with a strong interest in backend development, data structures and
algorithms, and scalable web app

  (score=0.238) – Alpha Intern, MMDU Coordinated campus events, managed volunteers, collaborated across
teams, and developed leadership, communication, and organizational skills. Projects
NutriScan – AI Powered Food Ingredient Analysis System  Buil

## Try Your Own Question
Run the cell below and type any question about the document.


In [15]:
user_question = input("Ask a question about the document: ")
ask(user_question)

Ask a question about the document: Who is Abhay?
Question: Who is Abhay?

Answer: Java Backend Developer

Retrieved context used:
  (score=0.293) Abhay Ranjan Java Backend Developer | MERN Stack Developer | Problem Solver +91 8294986686
| abhayranjan404@gmail.com | LinkedIn | GitHub | LeetCode Professional Summary Computer
Science undergraduate with a strong interest in backend development, data structures and
algorithms, and scalable web app

  (score=0.247) e working in collaborative environments through hackathons, projects, and student
leadership. Education Maharishi Markandeshwar (Deemed to be University) B.Tech in Computer
Science & Engineering (2023–2027) CGPA: 8.6/10 Experience Student Coordinator – Alpha
Intern, MMDU Coordinated campus events, m

  (score=0.101) ication (2025)  Frontend Web Developer Certification – Infosys Springboard (2025) 
Finalist – Internal Smart India Hackathon 2025  Participant – Hackureka Hackathon (GDSC)



## Improvements & Experiments
As noted in the project brief, this baseline pipeline can be extended in several ways:

- **Better chunking strategies**: sentence-aware or semantic chunking instead of fixed character windows
- **Different embedding models**: e.g. `all-mpnet-base-v2` for higher quality at the cost of speed
- **Hybrid search**: combine keyword search (BM25) with vector similarity for more robust retrieval
- **Re-ranking**: use a cross-encoder to re-rank the top retrieved chunks before generation
- **Different language models**: swap `flan-t5-base` for a larger model, or the Anthropic API, for higher quality answers


## Key Learnings

- How RAG systems combine **retrieval** (finding relevant information) and **generation**
  (producing a fluent, grounded answer)
- Why retrieval quality directly drives answer accuracy — a language model can only be as good
  as the context it's given
- How embeddings and vector databases (FAISS) enable fast semantic similarity search
- How to handle unstructured text data (PDFs) and turn it into a clean, queryable pipeline
- How to design a scalable AI pipeline made up of independent, swappable stages

## Conclusion
This project demonstrates how to build a system that can understand user queries, retrieve
relevant information from a custom document, and generate accurate, grounded answers.
RAG systems like this are widely used in chatbots, knowledge assistants, enterprise search
systems, and AI-powered documentation tools.
